# Feature engineering — STL

Este notebook constrói o dataset contendo as features de STL. Foi feito um notebook a parte pois a criação dessas features está demorando mais de 2 horas (Aprox. 140 minutos), então para ser possível criar outras features de forma mais rápidas, foi criado esse notebook a parte

**Entradas**
- `respiratory_hospitalization_time_series.parquet`

**Saída**
- `Data/GoldData/modelDatasetStlFeatures.parquet`

## Importações e configuração inicial

Nesta etapa são importadas as bibliotecas necessárias e definidos os parâmetros básicos de execução do notebook.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings('ignore')

## Caminhos dos dados e carregamento das bases

Aqui são definidos os caminhos das duas bases utilizadas no processo e realizado o carregamento inicial dos dados em memória.


In [ ]:
BASE_DIR = Path('../../Data')

PATH_INTERNACOES = 'https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/DataSus/respiratory_hospitalization_time_series.parquet'

PATH_OUTPUT = BASE_DIR / 'GoldData' / 'modelDatasetStlFeatures.parquet'

# Carrega as bases de dados
df_internacoes = pd.read_parquet(PATH_INTERNACOES)

print(f'Serie de internacoes: {df_internacoes.shape}')
print(f'  Periodo: {df_internacoes["data_dia"].min()} a {df_internacoes["data_dia"].max()}')

Serie de internacoes: (4018, 3)
  Periodo: 2008-01-01 00:00:00 a 2018-12-31 00:00:00

Serie de qualidade do ar: (2557, 12)
  Periodo: 2012-01-01 00:00:00 a 2018-12-31 00:00:00


## Preparação das datas e tratamento inicial

As colunas de data são padronizadas para o mesmo formato. As duas fontes são mantidas **separadas** nesta etapa:

- **`df_intern`** — série de internações completa (2008–2018). Todas as features endógenas (lags, janelas móveis, STL, calendário) são calculadas sobre essa série, aproveitando o histórico de 2008–2011 como contexto temporal.
- **`df_air`** — série de qualidade do ar (2012–2018). As features atmosféricas são calculadas aqui.

O **merge entre as duas fontes ocorre apenas na montagem do dataset final**, de modo que os dados de 2008–2011 contribuem como histórico para as features endógenas sem serem descartados antes do cálculo.



In [ ]:
df_internacoes['data'] = pd.to_datetime(df_internacoes['data_dia']).dt.normalize()

# ---------------------------------------------------------------------------
# DataFrame de internacoes: cobre toda a serie historica (2008-2018).
# Features endogenas serao calculadas aqui, aproveitando 2008-2011 como
# historico para lags longos (lag_365) e janela STL (~3 anos).
# ---------------------------------------------------------------------------
df_intern = (
    df_internacoes[['data', 'num_internacoes']]
    .drop_duplicates(subset='data')
    .sort_values('data')
    .reset_index(drop=True)
)

print(f'Serie de internacoes (df_intern): {df_intern.shape}')
print(f'  Periodo: {df_intern["data"].min().date()} a {df_intern["data"].max().date()}')

# Verifica continuidade
datas_esperadas = pd.date_range(df_intern['data'].min(), df_intern['data'].max(), freq='D')
datas_faltantes = datas_esperadas.difference(df_intern['data'])
print(f'  Dias faltantes: {len(datas_faltantes)}')
if len(datas_faltantes) > 0:
    print(f'  Primeiros faltantes: {sorted(datas_faltantes)[:5]}')

Serie de internacoes (df_intern): (4018, 2)
  Periodo: 2008-01-01 a 2018-12-31
  Dias faltantes: 0

Serie de qualidade do ar (df_air): (2557, 12)
  Periodo: 2012-01-01 a 2018-12-31

NaN em variaveis atmosfericas (antes -> depois da interpolacao):
  no: 23 -> 0
  no2: 23 -> 0
  so2: 23 -> 0
  pm2_5: 41 -> 0
  nox: 23 -> 0
  ur: 23 -> 0
  temp: 23 -> 0
  o3: 23 -> 0
  co: 23 -> 0
  pm10: 23 -> 0


## Definição da variável alvo

O alvo do modelo é o número de internações no dia corrente (`D0`). A partir daqui, todas as features são construídas para prever esse valor utilizando apenas informações observáveis até o instante da previsão.


In [ ]:
df_intern['target'] = df_intern['num_internacoes'].copy()

print('Alvo (target) — estatisticas descritivas:')
print(df_intern['target'].describe())

Alvo (target) — estatisticas descritivas:
count    4018.000000
mean       42.037830
std        20.261867
min         6.000000
25%        27.000000
50%        38.000000
75%        53.000000
max       171.000000
Name: target, dtype: float64


## Features de sazonalidade — decomposição STL

Esta seção extrai features baseadas em decomposição STL (*Seasonal and Trend decomposition using Loess*) com periodicidade anual (365 dias).

Para garantir a ausência de vazamento temporal (*lookahead bias*), o STL é ajustado em uma **janela móvel de ≈ 3 anos (1 095 dias)** de dados históricos terminando em `t-1`, excluindo o dia corrente. As features representam:

| Feature | Descrição |
|---|---|
| `stl365_seasonal` | Componente sazonal em `t-1` — proxy do ciclo anual no dia D0 |
| `stl365_resid_vol30` | Desvio padrão dos resíduos dos últimos 30 dias — captura instabilidade recente |
| `stl365_season_amp` | Amplitude sazonal dos últimos 365 dias — mede a intensidade do ciclo anual |

> **Nota sobre desempenho**: o cálculo envolve um ajuste STL por linha do dataset e pode levar alguns minutos.


In [ ]:
# =========================
# Parâmetros STL otimizados
# =========================
STL_PERIOD = 365                  # periodicidade anual
STL_WINDOW = 365 * 3              # janela móvel (~3 anos)
MIN_HISTORY = 2 * STL_PERIOD      # mínimo de histórico para rodar STL (~2 anos)

RESID_VOL_WINDOW = 30             # janela para volatilidade dos resíduos
SEASON_AMP_WINDOW = 365           # janela para amplitude sazonal

SEASONAL_WINDOW = 13              # LOESS sazonal (ímpar)
TREND_WINDOW = int(1.5 * STL_PERIOD)  # tendência mais suave (~547)

if SEASONAL_WINDOW % 2 == 0:
    SEASONAL_WINDOW += 1
if TREND_WINDOW % 2 == 0:
    TREND_WINDOW += 1


def compute_stl_rolling_features(
    series: pd.Series,
    window: int = STL_WINDOW,
    period: int = STL_PERIOD,
    min_history: int = MIN_HISTORY
) -> pd.DataFrame:
    """
    Calcula features de sazonalidade via decomposição STL com janela móvel (rolling),
    utilizando ESTRITAMENTE apenas dados passados (sem lookahead bias).

    Para o índice i (= D0), a janela usada é series.iloc[max(0, i-window) : i],
    ou seja, [t-window, t-1]. O valor de D0 (index i) NUNCA entra no ajuste STL.
    As features extraídas são referenciadas ao último ponto da janela (t-1).

    Com a série estendida a partir de 2008, o STL já terá ~2 anos de histórico
    disponível por volta de 2010, e a janela completa de 3 anos a partir de 2011.
    Para dados de 2012+ (que entrarão no dataset final após o merge), o STL terá
    sempre histórico completo e de qualidade.

    Features extraídas (todas calculadas sobre [t-window, t-1]):
    - stl365_seasonal    : componente sazonal no último ponto da janela (t-1)
    - stl365_resid_vol30 : desvio padrão dos últimos 30 resíduos da janela
    - stl365_season_amp  : amplitude sazonal (max - min) dos últimos 365 pontos
    """
    n = len(series)

    stl_seasonal    = np.full(n, np.nan)
    stl_resid_vol30 = np.full(n, np.nan)
    stl_season_amp  = np.full(n, np.nan)

    print(f'Computando STL rolling...')
    print(f'window={window} | min_history={min_history} | period={period}')
    print(f'seasonal={SEASONAL_WINDOW} | trend={TREND_WINDOW} | robust=True')

    for i in range(n):
        # Janela estritamente historica: [i-window, i-1]
        # series.iloc[start:i] exclui o indice i (= D0) — sem leakage.
        start = max(0, i - window)
        seg = series.iloc[start:i]      # <-- i excluido: janela termina em t-1

        if len(seg) < min_history:
            continue

        try:
            result = STL(
                seg,
                period=period,
                seasonal=SEASONAL_WINDOW,
                trend=TREND_WINDOW,
                robust=True
            ).fit()

            # Componente sazonal no ultimo ponto da janela (t-1)
            stl_seasonal[i] = result.seasonal.iloc[-1]

            # Volatilidade dos residuos nos ultimos RESID_VOL_WINDOW dias da janela
            k_res = min(RESID_VOL_WINDOW, len(result.resid))
            stl_resid_vol30[i] = result.resid.iloc[-k_res:].std(ddof=0)

            # Amplitude sazonal nos ultimos SEASON_AMP_WINDOW dias da janela
            k = min(SEASON_AMP_WINDOW, len(result.seasonal))
            last_season = result.seasonal.iloc[-k:]
            stl_season_amp[i] = last_season.max() - last_season.min()

        except Exception:
            pass

        if i % 300 == 0 and i > 0:
            pct = i / n * 100
            print(f'  {i:>5}/{n} ({pct:.0f}%)')

    print(f'  {n}/{n} (100%) — concluído.')

    return pd.DataFrame(
        {
            'stl365_seasonal':    stl_seasonal,
            'stl365_resid_vol30': stl_resid_vol30,
            'stl365_season_amp':  stl_season_amp,
        },
        index=series.index,
    )


# =========================
# Aplicação sobre df_intern
# =========================
serie_internacoes = df_intern['num_internacoes'].reset_index(drop=True)

df_stl = compute_stl_rolling_features(serie_internacoes)

df_intern[['stl365_seasonal', 'stl365_resid_vol30', 'stl365_season_amp']] = df_stl.values

# Resumo
stl_cols = ['stl365_seasonal', 'stl365_resid_vol30', 'stl365_season_amp']
print('\nFeatures STL criadas:')
for c in stl_cols:
    n_nulos = df_intern[c].isnull().sum()
    print(f'  {c}: {n_nulos} NaNs ({n_nulos / len(df_intern) * 100:.1f}%)')


Computando STL rolling...
window=1095 | min_history=730 | period=365
seasonal=13 | trend=547 | robust=True
    900/4018 (22%)
   1200/4018 (30%)
   1500/4018 (37%)
   1800/4018 (45%)
   2100/4018 (52%)
   2400/4018 (60%)
   2700/4018 (67%)
   3000/4018 (75%)
   3300/4018 (82%)
   3600/4018 (90%)
   3900/4018 (97%)
  4018/4018 (100%) — concluído.

Features STL criadas:
  stl365_seasonal: 730 NaNs (18.2%)
  stl365_resid_vol30: 730 NaNs (18.2%)
  stl365_season_amp: 730 NaNs (18.2%)


## Salvamento do dataset

Por fim, o dataframe final é salvo em formato `parquet` na camada `GoldData`, ficando pronto para uso na etapa de modelagem.


In [ ]:
PATH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Salva em formato parquet (eficiente e preserva tipos)
df_intern_final = df_intern[['data', 'stl365_seasonal', 'stl365_resid_vol30', 'stl365_season_amp']]
df_intern_final.to_parquet(PATH_OUTPUT, index=False)

file_size_kb = PATH_OUTPUT.stat().st_size / 1024
print(f'Dataset salvo em: {PATH_OUTPUT}')
print(f'  Shape: {df_intern_final.shape}')
print(f'  Tamanho: {file_size_kb:.1f} KB')

Dataset salvo em: ..\..\Data\GoldData\modelDataset.parquet
  Shape: (2387, 37)
  Tamanho: 383.5 KB
